# Declared and operative builders

This notebook shows what a method block does to a stage, on the consumption and portfolio stages of the portfolio–consumption application. Each stage file writes the maximization alone, with no marginal value and no candidate equation. The methods file names a block file, written in the language of the stage file, whose blocks compute the policy and state the derivatives of the values. Binding it with `methodize` changes the operative builders, `stage.builder`, which hold what the method adds; the declared builders, `stage.declared`, the operators the stage file's blocks denote, stay as they were. The stages are then joined in the two orders of a period, and the notebook shows the requirement that the endogenous grid method places on the stage that supplies the consumption stage's continuation. The notebook constructs and inspects; it solves nothing.

The files it reads are those of `applications/cons-port/declaration/`: `stages/cons/cons.bl`, `stages/port/port.bl`, `methods/cons.yml`, `methods/cons_egm.bl`, `methods/port.yml`, `methods/port_foc.bl`, `methods/port_without_marginals.yml` and the two recipes `spec.cons-port.yml` and `spec.port-cons.yml`, with the two recipes under `notebooks/without-portfolio-marginals/`.

The first cell imports the set-up module `checkout.py` beside this notebook, the module of the tutorials collection: it finds the checkout, puts its package first on the path and sets the working directory to the repository root, so that every path below is the path the repository gives.

In [1]:
import checkout
from bellman.stage import elaborate_stage, load_stage
from bellman.trellis import assemble_trellis, derivative_missing

## 1 The consumption stage file

The household arrives with resources $\mathrm m$, consumes $\mathrm c$ out of them and saves $\mathrm a = \mathrm m_d - \mathrm c$. The policy block writes the argmax of $u(c) + V_\succ(a)$ over the feasible set $\Gamma(m_d) = (0, m_d]$ and nothing else.

In [2]:
print(open("applications/cons-port/declaration/stages/cons/cons.bl").read())

@stage: cons
Xm := ℝ₊₊
Xa := ℝ₊
ρ : (1, inf)
𝑢(c : ℝ₊₊) := c^(1-ρ)/(1-ρ)
Γ(x : ℝ₊₊) := (0, x]

[id=dcsn_to_cntn, (m : Xm, c : ℝ₊₊ @ !ctrl) : !dcsn -> (a : Xa) : !cntn] {
    a = m - c
}

[id=cntn_to_dcsn, (V[>] : ℝ @ !vlu, c) : !cntn -> (V : ℝ @ !vlu) : !dcsn] {
    V = 𝑢(c) + V[>]
}

[id=policy, (V[>]) : !cntn -> (c) : !dcsn] {
    c = argmax_{cp : Γ(m)}(𝑢(cp) + V[>]) :: [id=optimality]
}



## 2 The two stages

`elaborate_stage` reads a stage file into a stage, with nothing bound.

In [3]:
cons = elaborate_stage(load_stage("applications/cons-port/declaration/stages/cons/cons.bl"))

In [4]:
cons

ElaboratedStage('cons', symbolic)

The portfolio stage chooses the risky share $\varsigma$ of the savings before the return $\Psi$ and income $\theta$ are drawn; its policy block also writes the argmax alone.

In [5]:
port = elaborate_stage(load_stage("applications/cons-port/declaration/stages/port/port.bl"))

In [6]:
port

ElaboratedStage('port', symbolic)

## 3 The declared policy builder

`cons.declared` answers the declared builders. The policy builder's card lists its four operations: the Pullback of the continuation value $V_\succ$ along the law of motion, the Aggregation that forms the objective, the Argmax over the feasible set, and the Evaluation, the identity into the declared set of policies. With nothing bound, `cons.builder.policy` is the same builder.

In [7]:
cons.declared.policy.denote()

STAGE cons / POLICY BUILDER policy
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Signature
  in   𝑉≻  : Xa → ℝ
       ──────────────────────────────────────────────────────────────────
  out  𝑐   : Xm → ℝ₊₊,  𝑐(m) ∈ Γ(m)

Operations
  1  Pullback
     in: 𝑉≻                             out: 𝑉≻ ∘ 𝑔≻
     ────────────────────────────────────────────────────────────────────
     rule: (𝑉≻ ∘ 𝑔≻)(m, c) = 𝑉≻(𝑔≻(m, c))

  2  optimality:Aggregation
     in: (1)                            out: (m, c) ↦ 𝑢(c) + 𝑉≻(𝑔≻(m, c))
     ────────────────────────────────────────────────────────────────────
     rule: (m, c) ↦ 𝑢(c) + 𝑉≻(𝑔≻(m, c))

  3  optimality:Argmax
     in: (2)                            out: 𝑐
     ────────────────────────────────────────────────────────────────────
     rule: 𝑐(m) ∈ argmax_{cp ∈ Γ(m)} [𝑢(cp) + 𝑉≻(𝑔≻(m, cp))]

  4  Evaluation
     in: 𝑐                              out: 𝑐
     ───────────────────────────────────────────────────────────

The decision builder maps the continuation value to the decision value. It states no derivative of its value, since the stage file writes no derivative line.

In [8]:
cons.declared["cntn", "dcsn"].denote()

STAGE cons / BACKWARD BUILDER cntn → dcsn (cntn_to_dcsn)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Signature
  in   𝑉≻  : Xa → ℝ
       ───────────────────────────────────────────────────
  out  𝑉   : Xm → ℝ

At
  𝑐  : Xm → ℝ₊₊,  𝑐(m) ∈ Γ(m)

Operations
  1  Pullback
     in: 𝑉≻                             out: 𝑉≻ ∘ 𝑔≻(·, 𝑐)
     ─────────────────────────────────────────────────────
     rule: (𝑉≻ ∘ 𝑔≻(·, 𝑐))(m) = 𝑉≻(𝑔≻(m, 𝑐(m)))

  2  Aggregation
     in: (1)                            out: 𝑉
     ─────────────────────────────────────────────────────
     rule: 𝑉(m) = 𝑢(𝑐(m)) + 𝑉≻(𝑔≻(m, 𝑐(m)))

  3  Evaluation
     in: 𝑉                              out: 𝑉
     ─────────────────────────────────────────────────────
     rule: 𝑉 : Xm → ℝ

Conditions
  Aggregation  𝑉(m) ∈ ℝ
──────────────────────────────────────────────────────────────────────────────
Source  cons.bl
  13  V = 𝑢(c) + V[>]

Definitions referenced
  Xm    cons:set:Xm
  Xa    cons:set:Xa

## 4 The block file

The block file `cons_egm.bl` writes the marginal utility of the stage's $u$ and its inverse in closed form, $u'(c) = c^{-\rho}$ and $(u')^{-1}(x) = x^{-1/\rho}$, and holds two blocks. The block `egm` computes the policy by the endogenous grid method: the candidate consumption at each saving with its Kuhn–Tucker clause, the resources at which that saving is chosen, and the policy read on those resources. The block `dcsn_marginal` is the marginal block of the decision value builder: the envelope condition $D_{m}V = u'(c)$, the derivative of the decision value in the resources.

In [9]:
print(open("applications/cons-port/declaration/methods/cons_egm.bl").read())

@methods: cons
𝑑𝑢(c : ℝ₊₊) := c^(-ρ)
𝑑𝑢𝑖𝑛𝑣(x : ℝ₊₊) := x^(-1/ρ)

[id=egm, (V[>] : ℝ @ !vlu) : !cntn -> (c : ℝ₊₊ @ !ctrl) : !dcsn] {
    c[>]   <= 𝑑𝑢𝑖𝑛𝑣(D_{a}V[>]) ⟂ a >= 0    :: [id=InvEuler]
    m[>]   = a + c[>]                        :: [id=grid]
    c      = c[>]                            :: [id=identification]
}






[id=dcsn_marginal, (c) : !cntn -> (V) : !dcsn] {
    D_{m}V = 𝑑𝑢(c)                           :: [id=envelope]
}



## 5 The methods file

Two lines name the block file: `!egm` on the policy builder, with the application's handler `cons-port:egm` beside the block, and `!marginals` on the decision value builder. The last line binds a representation of the stated derivative on the derivative line's id, `cntn_to_dcsn.envelope`.

In [10]:
print(open("applications/cons-port/declaration/methods/cons.yml").read())

stage: cons
# The consumption stage's methods. The stage file writes the maximization alone; the block file cons_egm.bl,
# beside this file, opens the policy builder with the endogenous-grid method and states the derivative of the
# decision value, the envelope condition. The function egm of the application's tool,
# applications/cons-port/code/cons_port_tool.py, realizes the method. The value is represented on the grid of
# resources the settings name.
methods:
  - on: policy
    method: !egm
    block: cons_egm.bl
    handler: cons-port:egm
  - on: cntn_to_dcsn
    method: !marginals
    block: cons_egm.bl
  - on: cntn_to_dcsn.envelope
    method: !Cartesian
    settings: {orders: {m: n_m}, bounds: {m: [m_min, m_max]}}



    ## Issues for aAS 24 oct 2026: 1) why evaluate was envelope, 2) how can i declare different methods ie polynomial spline etc. something on the the interpolated V ( should be fully flexible as to how the dV is created if at all). 
    # Answer (26 October 2026, A

## 6 Methodization with the block

`methodize` binds the methods file to the stage it is called on and returns that stage.

In [11]:
cons.methodize("applications/cons-port/declaration/methods/cons.yml")

ElaboratedStage('cons', methods)

`methods` lists the bindings by the place each stands: the block on the policy builder, the marginal block at the Evaluation of the decision value builder, and the representation of the stated derivative beside it.

In [12]:
cons.methods

policy                    !egm   block cons_egm.bl
cntn_to_dcsn:Evaluation   !marginals   block cons_egm.bl
cntn_to_dcsn:Evaluation   !Cartesian

## 7 The policy builder, declared and operative

The declared policy builder is the one printed in section 3, line for line.

In [13]:
cons.declared.policy.denote()

STAGE cons / POLICY BUILDER policy
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Signature
  in   𝑉≻  : Xa → ℝ
       ──────────────────────────────────────────────────────────────────
  out  𝑐   : Xm → ℝ₊₊,  𝑐(m) ∈ Γ(m)

Operations
  1  Pullback
     in: 𝑉≻                             out: 𝑉≻ ∘ 𝑔≻
     ────────────────────────────────────────────────────────────────────
     rule: (𝑉≻ ∘ 𝑔≻)(m, c) = 𝑉≻(𝑔≻(m, c))

  2  optimality:Aggregation
     in: (1)                            out: (m, c) ↦ 𝑢(c) + 𝑉≻(𝑔≻(m, c))
     ────────────────────────────────────────────────────────────────────
     rule: (m, c) ↦ 𝑢(c) + 𝑉≻(𝑔≻(m, c))

  3  optimality:Argmax
     in: (2)                            out: 𝑐
     ────────────────────────────────────────────────────────────────────
     rule: 𝑐(m) ∈ argmax_{cp ∈ Γ(m)} [𝑢(cp) + 𝑉≻(𝑔≻(m, cp))]

  4  Evaluation
     in: 𝑐                              out: 𝑐
     ───────────────────────────────────────────────────────────

The operative policy builder reads the derivative of the continuation value, $D_{a}V_\succ$, by the derivative term of its candidate line. Its card prints the stage file's argmax under `Declared`, the statement the construction is to reach, then the block's rows, the conditions by name, the method with its handler and block file, and the lines of both files under `Source`.

In [14]:
cons.builder.policy.denote()

STAGE cons / POLICY BUILDER policy
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Signature
  in   𝑉≻  : Xa → ℝ
       ────────────────────────────────────────
  out  𝑐   : Xm → ℝ₊₊,  𝑐(m) ∈ Γ(m)

Declared
  𝑐(m) ∈ argmax_{cp ∈ Γ(m)} [𝑢(cp) + 𝑉≻(𝑔≻(m, cp))]

Operations
  1  InvEuler:Aggregation
     in: 𝑉≻                             out: 𝑐≻
     ──────────────────────────────────────────
     rule: 𝑐≻(a) ≤ 𝑑𝑢𝑖𝑛𝑣(D_{a}𝑉≻(a)) ⟂ a ≥ 0

  2  grid:Aggregation
     in: 𝑐≻                             out: 𝑚≻
     ──────────────────────────────────────────
     rule: 𝑚≻(a) = a + 𝑐≻(a)

  3  identification:Pullback
     in: 𝑐≻, 𝑚≻                         out: 𝑐
     ──────────────────────────────────────────
     rule: 𝑐(𝑚≻(a)) = 𝑐≻(a)                                          on 𝑚≻(Xa)

  4  Evaluation
     in: 𝑐                              out: 𝑐
     ──────────────────────────────────────────
     rule: 𝑐 : Xm → ℝ₊₊,  𝑐(m) ∈ Γ(m)

Conditions
  grid:Aggregatio

## 8 The decision builder, declared and operative

The declared decision builder is unchanged by the binding.

In [15]:
cons.declared["cntn", "dcsn"].denote()

STAGE cons / BACKWARD BUILDER cntn → dcsn (cntn_to_dcsn)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Signature
  in   𝑉≻  : Xa → ℝ
       ───────────────────────────────────────────────────
  out  𝑉   : Xm → ℝ

At
  𝑐  : Xm → ℝ₊₊,  𝑐(m) ∈ Γ(m)

Operations
  1  Pullback
     in: 𝑉≻                             out: 𝑉≻ ∘ 𝑔≻(·, 𝑐)
     ─────────────────────────────────────────────────────
     rule: (𝑉≻ ∘ 𝑔≻(·, 𝑐))(m) = 𝑉≻(𝑔≻(m, 𝑐(m)))

  2  Aggregation
     in: (1)                            out: 𝑉
     ─────────────────────────────────────────────────────
     rule: 𝑉(m) = 𝑢(𝑐(m)) + 𝑉≻(𝑔≻(m, 𝑐(m)))

  3  Evaluation
     in: 𝑉                              out: 𝑉
     ─────────────────────────────────────────────────────
     rule: 𝑉 : Xm → ℝ

Conditions
  Aggregation  𝑉(m) ∈ ℝ
──────────────────────────────────────────────────────────────────────────────
Source  cons.bl
  13  V = 𝑢(c) + V[>]

Definitions referenced
  Xm    cons:set:Xm
  Xa    cons:set:Xa

The operative decision builder states the derivative of the value it writes: the envelope line stands at its Evaluation beside the representation, the signature is unchanged, and the `Source` section quotes the line from the block file.

In [16]:
cons.builder["cntn", "dcsn"].denote()

STAGE cons / BACKWARD BUILDER cntn → dcsn (cntn_to_dcsn)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Signature
  in   𝑉≻  : Xa → ℝ
       ───────────────────────────────────────────────────
  out  𝑉   : Xm → ℝ

At
  𝑐  : Xm → ℝ₊₊,  𝑐(m) ∈ Γ(m)

Operations
  1  Pullback
     in: 𝑉≻                             out: 𝑉≻ ∘ 𝑔≻(·, 𝑐)
     ─────────────────────────────────────────────────────
     rule: (𝑉≻ ∘ 𝑔≻(·, 𝑐))(m) = 𝑉≻(𝑔≻(m, 𝑐(m)))

  2  Aggregation
     in: (1)                            out: 𝑉
     ─────────────────────────────────────────────────────
     rule: 𝑉(m) = 𝑢(𝑐(m)) + 𝑉≻(𝑔≻(m, 𝑐(m)))

  3  Evaluation
     in: 𝑉                              out: 𝑉
     ─────────────────────────────────────────────────────
     rule: 𝑉 : Xm → ℝ
           D_{m}𝑉(m) = 𝑑𝑢(𝑐(m))

Conditions
  Aggregation  𝑉(m) ∈ ℝ

Method
  !marginals   block cons_egm.bl                       cntn_to_dcsn:Evaluation

Method
  !Cartesian   handler                               

## 9 The conditions

`obligations()` lists the condition of every rule applied, by its name. Among them are `egm-eligibility`, under which the policy the block computes satisfies the declared argmax, `function-argument-domain-inclusion`, that the derivative the candidate line passes to the inverse of the marginal utility lies in its domain, and `marginal-reading`, the reading of the author's derivative line. The package checks none of them.

In [17]:
cons.obligations()

finite-real-arithmetic               The operands of Add at policy.optimality must be finite real numbers.
aggregation-admissibility            V(m) ∈ ℝ
finite-real-arithmetic               The operands of Add at policy must be finite real numbers.
finite-real-arithmetic               The operands of Add at policy.grid must be finite real numbers.
aggregation-admissibility            m≻(a) ∈ Xm
marginal-reading                     the right side of each derivative line equals the derivative, in the named coordinate, of the value the builder writes, under the envelope conditions: concavity of the continuation value, interiority of the maximizer or the multiplier's term at a binding constraint, differentiability of the reward in the control, a measurable selection of the policy, and differentiability of the value at the point read; listed and not checked
range-compatibility                  m − c(m) ∈ Xa   [established]
function-argument-domain-inclusion   The argument for x of cons:func

## 10 The portfolio stage's block file

The block `foc` computes the share from its first-order condition, the expected excess return weighted by the derivative of the continuation value; the marginal block states the derivative of the decision value in capital.

In [18]:
print(open("applications/cons-port/declaration/methods/port_foc.bl").read())

@stage: port

[id=foc, (V[>]) : !cntn -> ς] {
    ς = solve_{ς}(E_{Ψ,θ}(D_{w}V[>]*(Ψ - R)))       :: [id=euler]
}

[id=dcsn_marginal, (V[>], ς) : !cntn -> (V) : !dcsn] {
    D_{k}V = E_{Ψ,θ}(D_{w}V[>]*(ς*Ψ + (1-ς)*R))     :: [id=envelope]
}



In [19]:
port.methodize("applications/cons-port/declaration/methods/port.yml")

ElaboratedStage('port', methods)

The declared policy builder of the portfolio stage holds the argmax over the share; the operative one holds the first-order condition, under the method `!time-iteration`, with the argmax printed under `Declared`. The handler `cons-port:share_search` beside the block chooses the share by the search over the declared objective, as the application's tool realizes it.

In [20]:
port.declared.policy.denote()

STAGE port / POLICY BUILDER policy
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Signature
  in   𝑉≻  : ℝ₊₊ → ℝ
       ─────────────────────────────────────────────
  out  𝜍   : ℝ₊ → [0, 1],  𝜍(k) ∈ Γ_ς(k)

Operations
  1  Pullback
     in: 𝑉≻                             out: 𝑉≻ ∘ 𝑔≻
     ───────────────────────────────────────────────
     rule: (𝑉≻ ∘ 𝑔≻)(k, ς, Ψ, θ) = 𝑉≻(𝑔≻(k, ς, Ψ, θ))

  2  objective:Expectation:1
     in: (1)
     out: (k, ς) ↦ E_{Ψ, θ}[𝑉≻(𝑔≻(k, ς, Ψ, θ))]
     ───────────────────────────────────────────────
     rule: (2)(k, ς) = E_{Ψ, θ}[𝑉≻(𝑔≻(k, ς, Ψ, θ))]

  3  objective:Argmax
     in: (2)                            out: 𝜍
     ───────────────────────────────────────────────
     rule: 𝜍(k) ∈ argmax_{ς ∈ Γ_ς(k)} [E_{Ψ, θ}[𝑉≻(𝑔≻(k, ς, Ψ, θ))]]

  4  Evaluation
     in: 𝜍                              out: 𝜍
     ───────────────────────────────────────────────
     rule: 𝜍 : ℝ₊ → [0, 1],  𝜍(k) ∈ Γ_ς(k)
───────────────────────────

In [21]:
port.builder.policy.denote()

STAGE port / POLICY BUILDER policy
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Signature
  in   𝑉≻  : ℝ₊₊ → ℝ
       ───────────────────────────────────────────────────
  out  𝜍   : ℝ₊ → [0, 1],  𝜍(k) ∈ Γ_ς(k)

Declared
  𝜍(k) ∈ argmax_{ς ∈ Γ_ς(k)} [E_{Ψ, θ}[𝑉≻(𝑔≻(k, ς, Ψ, θ))]]

Operations
  1  Pullback
     in: 𝑉≻                             out: 𝑉≻ ∘ 𝑔≻
     ─────────────────────────────────────────────────────
     rule: (𝑉≻ ∘ 𝑔≻)(k, ς, Ψ, θ) = 𝑉≻(𝑔≻(k, ς, Ψ, θ))

  2  euler:Aggregation:1
     in: (1)
     out: (k, ς, Ψ, θ) ↦ D_{w}𝑉≻(𝑔≻(k, ς, Ψ, θ)) · (Ψ − R)
     ─────────────────────────────────────────────────────
     rule: (k, ς, Ψ, θ) ↦ D_{w}𝑉≻(𝑔≻(k, ς, Ψ, θ)) · (Ψ − R)

  3  euler:Expectation:1
     in: (2)                            out: r₁
     ─────────────────────────────────────────────────────
     rule: r₁ = E_{Ψ, θ}[D_{w}𝑉≻(𝑔≻(k, ς, Ψ, θ)) · (Ψ − R)]

  4  euler:Aggregation
     in: r₁                             out: 𝜍
     ───────

## 11 The two orders

Each recipe joins the consumption, discount and portfolio stages into a period and repeats it six times; `assemble_trellis` reads the recipe and binds every stage occurrence with its calibration, methods and settings. In the first order consumption is chosen first and the share on the savings.

In [22]:
cons_first = assemble_trellis("applications/cons-port/declaration", "spec.cons-port.yml")

The consumption stage's block reads $D_{a}V_\succ$, so the value that the stage supplying its continuation exports must carry its derivative in the savings, and so on along the period. `derivative_missing` lists each derivative reading that no route meets, naming the stage, the value and the coordinate. Every value here carries its derivative, by a stated line or by its representation, and the list is empty.

In [23]:
derivative_missing(cons_first)

()

In the second order the share is chosen first, on the arrival resources, and consumption on the resources the draws give.

In [24]:
port_first = assemble_trellis("applications/cons-port/declaration", "spec.port-cons.yml")

In [25]:
derivative_missing(port_first)

()

## 12 Without the portfolio stage's marginal block

The methods file `port_without_marginals.yml` is the portfolio stage's methods file without its `!marginals` line and without its `!Linear` representation, which would evaluate the first derivative of the represented value, so that the value the stage exports carries no derivative in capital.

In [26]:
print(open("applications/cons-port/declaration/methods/port_without_marginals.yml").read())

stage: port
# The portfolio stage's methods of port.yml, beside this file, without its !marginals line and without its
# !Linear representation, which would evaluate the first derivative of the represented value, so that the value
# the stage exports carries no derivative. The two recipes under notebooks/without-portfolio-marginals/ name
# this file, and the notebook notebooks/declared-and-operative.ipynb binds them to show the requirement that the
# consumption stage's endogenous grid method, which reads a derivative of its continuation value, places on
# the stage that supplies that value.
methods:
  - on: policy
    method: !time-iteration
    block: port_foc.bl
    handler: cons-port:share_search
  - on: E_Ψ_θ
    method: !gauss-hermite
    handler: cons-port:gauss_hermite
    settings: {n_nodes: [n_Ψ_nodes, n_θ_nodes]}
  - on: dcsn_to_cntn
    method: !monte-carlo
    settings: {n_paths: n_sim, seed: sim_seed}



The two recipes under `notebooks/without-portfolio-marginals/` are the folder's recipes with this file in place of `port.yml`. Consumption first:

In [27]:
cons_first_without = assemble_trellis("applications/cons-port/declaration", "../notebooks/without-portfolio-marginals/spec.cons-port.yml")

Each entry names the portfolio occurrence, its value and the coordinate `k`, the reading `D_{a}V[>]` of the consumption stage's block that reaches it through the discount stage's pass-through, and says what to bind. The stages bind and nothing is refused: a missing derivative is a requirement reported on the trellis, not a refusal of the method.

In [28]:
derivative_missing(cons_first_without)

(Diagnostic(rule='derivative-missing', address="periods[0].stages['port']", message="stage 'port' at periods[0].port supplies, through the pass-through of disc, the continuation of periods[0].cons, whose block 'policy' reads `D_{a}V[>]`, and its value `V` carries no derivative in k; state it with a derivative line of the builder that writes the value of stage 'port', in its stage file or in a marginal block bound with `!marginals`, or bind on that builder a representation whose row of the method table evaluates it", candidates=('port:function:dcsn:V', 'k'), source_entries=()),
 Diagnostic(rule='derivative-missing', address="periods[1].stages['port']", message="stage 'port' at periods[1].port supplies, through the pass-through of disc, the continuation of periods[1].cons, whose block 'policy' reads `D_{a}V[>]`, and its value `V` carries no derivative in k; state it with a derivative line of the builder that writes the value of stage 'port', in its stage file or in a marginal block bound

Portfolio first:

In [29]:
port_first_without = assemble_trellis("applications/cons-port/declaration", "../notebooks/without-portfolio-marginals/spec.port-cons.yml")

In [30]:
derivative_missing(port_first_without)

(Diagnostic(rule='derivative-missing', address="periods[0].stages['port']", message="stage 'port' at periods[0].port supplies, through the pass-through of disc, the continuation of periods[1].cons, whose block 'policy' reads `D_{a}V[>]`, and its value `V` carries no derivative in k; state it with a derivative line of the builder that writes the value of stage 'port', in its stage file or in a marginal block bound with `!marginals`, or bind on that builder a representation whose row of the method table evaluates it", candidates=('port:function:dcsn:V', 'k'), source_entries=()),
 Diagnostic(rule='derivative-missing', address="periods[1].stages['port']", message="stage 'port' at periods[1].port supplies, through the pass-through of disc, the continuation of periods[2].cons, whose block 'policy' reads `D_{a}V[>]`, and its value `V` carries no derivative in k; state it with a derivative line of the builder that writes the value of stage 'port', in its stage file or in a marginal block bound

## 13 The two states in the Workbench

Bellman Workbench shows a stage file beside the graph it denotes. The layers button at the foot of the graph switches the builders shown. Off, the graph and the cards are the declared builders: the policy builder of the consumption stage holds the argmax, and the details name the bound method.

![The consumption stage in Bellman Workbench with the layers button off: the declared builders](images/layers-button-off.png)

On, they are the operative builders: the decision builder's Evaluation states the derivative $D_{m}V$ under `!marginals` beside the representation `!Cartesian`, and the details of the policy builder print its signature, its declared line and its conditions. The two pictures were taken on 22 October 2026, before the derivative term replaced the marginal field, and show the earlier labels.

![The consumption stage in Bellman Workbench with the layers button on: the operative builders](images/layers-button-on.png)

### Observation

### Decision